## 1. Source Schema

### Question

What fields and data types are present in the Bronze lab-results feed?

### Purpose

Establish the source contract before defining the Silver grain,

normalization, data-quality, referential-integrity, or CDC rules.

In [0]:
%sql

DESCRIBE TABLE clinical_trial_intelligence.bronze.lab_results;

## 2. Sample Bronze Lab Result Records

### Question

What do representative Bronze lab-result records look like before
Silver transformation?

### Purpose

Inspect actual source values, identifier formats, laboratory test
codes and names, result values, units, reference ranges, abnormal
flags, collection dates, vendors, missing values, and ingestion
metadata before defining Silver business and data-quality rules.

In [0]:
%sql

SELECT *
FROM clinical_trial_intelligence.bronze.lab_results
LIMIT 50;

## 3. Dataset Profile and Completeness

### Question

What is the overall volume and completeness profile of the Bronze lab-results feed?

### Purpose

Establish the Bronze baseline before defining Silver transformation and data-quality rules. This identifies missing business keys, relationship identifiers, laboratory measurements, reference ranges, collection dates, rescued records, and the number of source files contributing to the dataset.

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT lab_result_id)
        AS distinct_lab_result_ids,

    SUM(
        CASE
            WHEN lab_result_id IS NULL
              OR TRIM(lab_result_id) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_lab_result_id,

    SUM(
        CASE
            WHEN subject_id IS NULL
              OR TRIM(subject_id) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_subject_id,

    SUM(
        CASE
            WHEN study_id IS NULL
              OR TRIM(study_id) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_study_id,

    SUM(
        CASE
            WHEN visit_id IS NULL
              OR TRIM(visit_id) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_visit_id,

    SUM(
        CASE
            WHEN lab_test_code IS NULL
              OR TRIM(lab_test_code) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_lab_test_code,

    SUM(
        CASE
            WHEN result_value IS NULL
            THEN 1 ELSE 0
        END
    ) AS missing_result_value,

    SUM(
        CASE
            WHEN result_unit IS NULL
              OR TRIM(result_unit) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_result_unit,

    SUM(
        CASE
            WHEN reference_low IS NULL
            THEN 1 ELSE 0
        END
    ) AS missing_reference_low,

    SUM(
        CASE
            WHEN reference_high IS NULL
            THEN 1 ELSE 0
        END
    ) AS missing_reference_high,

    SUM(
        CASE
            WHEN abnormal_flag IS NULL
              OR TRIM(abnormal_flag) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_abnormal_flag,

    SUM(
        CASE
            WHEN collection_date IS NULL
              OR TRIM(collection_date) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_collection_date,

    SUM(
        CASE
            WHEN _rescued_data IS NOT NULL
            THEN 1 ELSE 0
        END
    ) AS rescued_data_rows,

    COUNT(DISTINCT _source_file_name)
        AS source_file_count

FROM clinical_trial_intelligence.bronze.lab_results;

### Result

The Bronze `lab_results` dataset contains **65,006 rows** representing **65,006 distinct `lab_result_id` values**.

Key completeness findings:

- `lab_result_id`: **0 missing**
- `subject_id`: **0 missing**
- `study_id`: **0 missing**
- `visit_id`: **0 missing**
- `lab_test_code`: **0 missing**
- `result_value`: **692 missing**
- `result_unit`: **519 missing**
- `reference_low`: **0 missing**
- `reference_high`: **0 missing**
- `abnormal_flag`: **0 missing**
- `collection_date`: **0 missing**
- `_rescued_data`: **0 rows**
- Source files: **10**

The row count and distinct `lab_result_id` count are identical, indicating that no duplicate `lab_result_id` values are present in the currently observed Bronze dataset.

The only observed completeness issues are concentrated in `result_value` and `result_unit`.

### Conclusion

The Bronze lab-results feed is structurally complete for its business key and major relationship identifiers.

`lab_result_id` is a strong candidate business key because all **65,006** records contain a value and all **65,006** values are distinct.

No missing values are currently observed for `subject_id`, `study_id`, `visit_id`, `lab_test_code`, reference-range boundaries, `abnormal_flag`, or `collection_date`.

However, **692 records have no `result_value` and 519 records have no `result_unit`**. These records should not yet be classified as invalid because the clinical context of the missing values has not been established.

Further exploration is required to determine:

1. whether missing results are associated with particular tests, vendors, or source files;
2. whether missing units occur only when the corresponding result is also missing;
3. whether some laboratory tests legitimately permit a missing unit;
4. whether the missingness represents source-data quality problems requiring quarantine.

Therefore, no Silver rejection rule for `result_value` or `result_unit` should be introduced until these patterns are investigated.

## 4. Missing Result Value and Result Unit Investigation

### Question

How are missing `result_value` and `result_unit` values distributed, and do the two completeness issues overlap?

### Purpose

Determine whether missing laboratory results and units represent the same records or separate data-quality patterns.

This investigation also identifies whether the missing values are concentrated in particular laboratory tests, vendors, or source files before deciding whether these conditions should become blocking Silver data-quality rules.

In [0]:
%sql

SELECT
    CASE
        WHEN result_value IS NULL
         AND (result_unit IS NULL OR TRIM(result_unit) = '')
            THEN 'missing_value_and_unit'

        WHEN result_value IS NULL
            THEN 'missing_value_only'

        WHEN result_unit IS NULL
          OR TRIM(result_unit) = ''
            THEN 'missing_unit_only'

        ELSE 'complete'
    END AS completeness_pattern,

    COUNT(*) AS row_count,
    COUNT(DISTINCT lab_result_id) AS distinct_lab_results,
    COUNT(DISTINCT lab_test_code) AS distinct_lab_tests,
    COUNT(DISTINCT lab_vendor) AS distinct_vendors,
    COUNT(DISTINCT _source_file_name) AS source_file_count

FROM clinical_trial_intelligence.bronze.lab_results

GROUP BY
    CASE
        WHEN result_value IS NULL
         AND (result_unit IS NULL OR TRIM(result_unit) = '')
            THEN 'missing_value_and_unit'

        WHEN result_value IS NULL
            THEN 'missing_value_only'

        WHEN result_unit IS NULL
          OR TRIM(result_unit) = ''
            THEN 'missing_unit_only'

        ELSE 'complete'
    END

ORDER BY row_count DESC;

In [0]:
%sql

SELECT
    lab_test_code,
    lab_test_name,
    lab_vendor,

    COUNT(*) AS affected_rows,

    SUM(
        CASE
            WHEN result_value IS NULL
            THEN 1 ELSE 0
        END
    ) AS missing_result_value,

    SUM(
        CASE
            WHEN result_unit IS NULL
              OR TRIM(result_unit) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_result_unit,

    COUNT(DISTINCT _source_file_name)
        AS affected_source_files

FROM clinical_trial_intelligence.bronze.lab_results

WHERE
       result_value IS NULL
    OR result_unit IS NULL
    OR TRIM(result_unit) = ''

GROUP BY
    lab_test_code,
    lab_test_name,
    lab_vendor

ORDER BY
    affected_rows DESC,
    lab_test_code,
    lab_vendor;

### Result

The missingness investigation identified three distinct completeness patterns:

- **63,795 records** are complete.
- **692 records** have a missing `result_value` while `result_unit` remains populated.
- **519 records** have a missing `result_unit` while `result_value` remains populated.
- **0 records** have both `result_value` and `result_unit` missing.

The missing-value issue spans **8 laboratory tests, 2 vendors, and 8 source files**.

The missing-unit issue also spans **8 laboratory tests and both laboratory vendors**, and occurs across **5 source files**.

The test/vendor-level analysis shows that the affected records are distributed across multiple laboratory tests, including ALT, AST, HGB, potassium, creatinine, WBC, platelet count, and glucose.

The issues are also present for both `CENTRALAB` and `MEDPATH`, rather than being isolated to a single laboratory vendor.

### Conclusion

Missing `result_value` and missing `result_unit` represent two separate source-data quality patterns.

A missing `result_value` cannot be explained simply by the absence of a corresponding unit because all **692** affected records still contain a unit.

Similarly, all **519** records with a missing unit contain a laboratory result value.

The issues are distributed across multiple laboratory tests, vendors, and source files. Therefore, they are not attributable to one isolated test or one laboratory vendor based on the currently observed data.

Both conditions require explicit Silver handling, but they should not be treated as the same validation rule.

Before determining whether these records should be quarantined, the reference tables and laboratory-test semantics must be investigated to establish whether a result value and unit are mandatory for every supported test.

## 5. Laboratory Test and Unit Reference Coverage

### Question

Can every populated Bronze laboratory test and result-unit combination be resolved through the existing Silver reference tables?

### Purpose

Determine whether `ref_lab_test` and `ref_unit` provide sufficient coverage to standardize laboratory test codes and heterogeneous source-unit representations.

The investigation establishes whether unit normalization can be performed through reference-driven mappings rather than hardcoded transformations in the Silver pipeline.

In [0]:
%sql

SELECT *
FROM clinical_trial_intelligence.silver.ref_lab_test
ORDER BY lab_test_code;

In [0]:
%sql
SELECT *
FROM clinical_trial_intelligence.silver.ref_unit
ORDER BY lab_test_code, raw_unit;

In [0]:
%sql

SELECT
    UPPER(TRIM(l.lab_test_code)) AS lab_test_code,
    COUNT(*) AS bronze_rows,
    COUNT_IF(r.lab_test_code IS NULL) AS unmapped_rows

FROM clinical_trial_intelligence.bronze.lab_results l

LEFT JOIN clinical_trial_intelligence.silver.ref_lab_test r
    ON UPPER(TRIM(l.lab_test_code))
     = UPPER(TRIM(r.lab_test_code))

GROUP BY
    UPPER(TRIM(l.lab_test_code))

ORDER BY lab_test_code;

In [0]:
%sql

SELECT
    UPPER(TRIM(l.lab_test_code)) AS lab_test_code,
    l.result_unit AS raw_result_unit,
    COUNT(*) AS bronze_rows,
    COUNT_IF(u.raw_unit IS NULL) AS unmapped_rows

FROM clinical_trial_intelligence.bronze.lab_results l

LEFT JOIN clinical_trial_intelligence.silver.ref_unit u
    ON UPPER(TRIM(l.lab_test_code))
       = UPPER(TRIM(u.lab_test_code))
   AND UPPER(TRIM(l.result_unit))
       = UPPER(TRIM(u.raw_unit))

WHERE l.result_unit IS NOT NULL
  AND TRIM(l.result_unit) <> ''

GROUP BY
    UPPER(TRIM(l.lab_test_code)),
    l.result_unit

ORDER BY
    lab_test_code,
    raw_result_unit;

### Result

The laboratory reference tables provide complete mapping coverage for all recognized laboratory test codes.

Eight valid laboratory test codes were identified in the Bronze feed:

- ALT
- AST
- CREAT
- GLUC
- HGB
- K
- PLT
- WBC

All records associated with these eight test codes successfully resolve to `ref_lab_test`.

However, **377 Bronze records** contain the laboratory test code `XXX`. All 377 records are unmapped because `XXX` does not exist in the laboratory-test reference table.

Unit analysis shows that the source contains multiple representations of equivalent laboratory units. Examples include:

- `IU/L`, `U/L`, `UL`, and `u/l` for ALT/AST
- `MG/DL`, `mg / dL`, `mg/dL`, and `mg/dl` for creatinine
- `G/DL`, `g/dL`, `g/dl`, and `gm/dL` for hemoglobin
- `MMOL/L`, `mEq/L`, and `mmol/L` for potassium
- `10^9/L`, `K/uL`, and `x10E9/L` for platelet count

The `ref_unit` table provides mappings for these recognized test/unit combinations and also contains conversion factors where numerical conversion is required.

Examples include:

- Glucose: `mmol/L` → `mg/dL`, conversion factor **18**
- Hemoglobin: `g/L` → `g/dL`, conversion factor **0.1**
- Other observed equivalent unit representations use a conversion factor of **1**

All populated units belonging to the eight recognized laboratory tests resolve successfully through `ref_unit`.

The unmapped unit combinations are confined to records where `lab_test_code = 'XXX'`.

### Conclusion

The existing reference architecture is sufficient to perform reference-driven laboratory standardization for all recognized laboratory tests observed in the Bronze dataset.

Silver should therefore avoid hardcoded unit-normalization logic. Instead, laboratory results should be enriched using:

`ref_lab_test`
→ validates the laboratory test and supplies the expected standard unit and reference range.

`ref_unit`
→ maps the raw test/unit combination to the standard unit and supplies the required numerical conversion factor.

The standardized result can consequently be derived as:

**standardized_result_value = result_value × conversion_factor**

Records with `lab_test_code = 'XXX'` cannot be reliably interpreted because the test itself is unknown. The associated unit alone is insufficient to determine the clinical meaning of the measurement.

Therefore, the **377 records with `lab_test_code = 'XXX'` should be treated as blocking DQ failures and routed to quarantine** rather than inferred or force-mapped.

This establishes the Silver design principle:

**recognized test + recognized unit → standardize and retain**

**unknown test → quarantine**

## 6. Reference-Key Uniqueness

### Question

Does normalization of laboratory unit strings create duplicate lookup keys in the unit reference table?

### Purpose

Verify that the normalized `(lab_test_code, raw_unit)` lookup key is unique before using `ref_unit` in the Silver transformation.

A non-unique reference key would multiply Bronze records during the join and corrupt the Silver row grain.

In [0]:
%sql
SELECT
    UPPER(TRIM(lab_test_code)) AS lab_test_code,
    UPPER(TRIM(raw_unit)) AS normalized_raw_unit,
    COUNT(*) AS reference_rows

FROM clinical_trial_intelligence.silver.ref_unit

GROUP BY
    UPPER(TRIM(lab_test_code)),
    UPPER(TRIM(raw_unit))

HAVING COUNT(*) > 1

ORDER BY
    lab_test_code,
    normalized_raw_unit;

### Result

Normalization of `raw_unit` using `UPPER(TRIM())` creates duplicate lookup keys in the unit reference table.

Duplicate normalized keys were identified for:

- ALT + U/L → 2 reference rows
- AST + U/L → 2 reference rows
- CREAT + MG/DL → 3 reference rows
- GLUC + MG/DL → 3 reference rows
- HGB + G/DL → 3 reference rows
- K + MMOL/L → 2 reference rows

These duplicates arise because multiple raw textual representations become identical after case normalization.

For example, values such as `U/L` and `u/l` represent the same logical unit but collapse to the same normalized lookup key.

Therefore, directly joining Bronze laboratory records to the normalized `ref_unit` table would create a many-to-one reference match and could multiply laboratory-result rows.

### Conclusion

The unit reference table is clinically sufficient but its normalized lookup key is not unique.

Therefore, the Silver transformation must first construct a deterministic unit-reference lookup containing exactly one row per:

`(lab_test_code, normalized_raw_unit)`

before joining it to the laboratory-result stream.

The deduplicated reference must preserve:

- `lab_test_code`
- normalized raw unit
- `standard_unit`
- `conversion_factor`

This prevents reference-table duplication from changing the business grain of the laboratory dataset.

Silver must preserve:

**one source `lab_result_id` → at most one Silver laboratory-result record**

The original reference table should not be modified merely to support the Silver join. Deduplication should be handled inside the Silver transformation/reference-resolution logic.

## 7. Abnormal-Flag Consistency

### Question

Does the source-provided `abnormal_flag` agree with the laboratory result and its reported reference range?

### Purpose

Evaluate whether the source-provided abnormality indicator can be trusted directly in Silver.

A laboratory result should normally be considered outside its reported reference range when:

`result_value < reference_low`

or

`result_value > reference_high`

This analysis compares the source `abnormal_flag` with a range status derived independently from `result_value`, `reference_low`, and `reference_high`.

The result will determine whether Silver should retain the source abnormal flag as authoritative or derive a standardized abnormality indicator from the laboratory measurement and reference range.

In [0]:
%sql

SELECT
    abnormal_flag,

    CASE
        WHEN result_value IS NULL
            THEN 'missing_result'

        WHEN result_value < reference_low
          OR result_value > reference_high
            THEN 'outside_reference_range'

        ELSE 'within_reference_range'
    END AS derived_range_status,

    COUNT(*) AS row_count,
    COUNT(DISTINCT lab_result_id) AS distinct_lab_results,
    COUNT(DISTINCT lab_test_code) AS distinct_lab_tests,
    COUNT(DISTINCT _source_file_name) AS source_file_count

FROM clinical_trial_intelligence.bronze.lab_results

GROUP BY
    abnormal_flag,

    CASE
        WHEN result_value IS NULL
            THEN 'missing_result'

        WHEN result_value < reference_low
          OR result_value > reference_high
            THEN 'outside_reference_range'

        ELSE 'within_reference_range'
    END

ORDER BY
    abnormal_flag,
    derived_range_status;

### Result

The source-provided `abnormal_flag` is not consistently aligned with the laboratory result and its reported reference range.

The analysis identified:

- **50,178** records marked `N` whose results are within the reported reference range.
- **10,368** records marked `N` whose results are outside the reported reference range.
- **3,763** records marked `Y` whose results are outside the reported reference range.
- **5** records marked `Y` even though their results are within the reported reference range.
- **650** records marked `N` where `result_value` is missing.
- **42** records marked `Y` where `result_value` is missing.

Therefore, **10,373 populated laboratory results** have a direct disagreement between the source-provided abnormal flag and the abnormality derived from the reported result/reference range:

- 10,368 `N` records are outside the reference range.
- 5 `Y` records are within the reference range.

The inconsistencies are not isolated to a single laboratory test or source file. Records marked `N` but outside the reference range occur across all **9 observed test codes** and all **10 source files**.

The source `abnormal_flag` therefore cannot be treated as a reliable authoritative indicator of laboratory abnormality.

### Conclusion

Silver should not use the source-provided `abnormal_flag` as the authoritative abnormality classification.

For records with a valid standardized result and valid reference range, Silver should derive a standardized abnormality indicator from the measurement itself:

`standardized_result_value < reference_low`

or

`standardized_result_value > reference_high`

The original source `abnormal_flag` should be retained separately for lineage and auditability so that discrepancies between the source classification and the derived classification remain traceable.

Records with a missing `result_value` cannot be independently classified as within or outside the reference range. Therefore, their abnormality status should not be inferred solely from the source flag.

The Silver design should consequently distinguish between:

- the **source-provided abnormal flag**, retained for lineage;
- the **derived abnormal flag**, calculated from the standardized laboratory result and valid reference range; and
- a **flag discrepancy indicator**, identifying disagreement between source and derived classifications.

The observed disagreement should be treated as a source data-quality characteristic rather than automatically quarantining otherwise interpretable laboratory measurements.

## 8. Referential Integrity

### Question

Do the `subject_id`, `visit_id`, and `study_id` values in the Bronze laboratory-results feed resolve correctly to the corresponding clinical entities?

### Purpose

Validate the relationships between laboratory results and the core clinical-trial entities before defining the Silver transformation.

Each laboratory result should belong to:

- a known subject;
- a known visit;
- a known study;
- a visit belonging to the same subject recorded on the laboratory result; and
- a visit belonging to the same study recorded on the laboratory result.

This analysis identifies orphan laboratory records and cross-entity inconsistencies that could make a laboratory result clinically ambiguous or incorrectly attributed.%md


In [0]:
%sql

WITH labs AS (

    SELECT *
    FROM clinical_trial_intelligence.bronze.lab_results

),

subjects AS (

    SELECT DISTINCT
        subject_id,
        study_id

    FROM clinical_trial_intelligence.bronze.edc_subjects

    WHERE subject_id IS NOT NULL

),

visits AS (

    SELECT DISTINCT
        visit_id,
        subject_id,
        study_id

    FROM clinical_trial_intelligence.bronze.edc_visits

    WHERE visit_id IS NOT NULL

),

checked AS (

    SELECT
        l.lab_result_id,

        l.subject_id AS lab_subject_id,
        l.visit_id   AS lab_visit_id,
        l.study_id   AS lab_study_id,

        s.subject_id AS matched_subject_id,

        v.visit_id   AS matched_visit_id,
        v.subject_id AS visit_subject_id,
        v.study_id   AS visit_study_id

    FROM labs l

    LEFT JOIN subjects s
        ON l.subject_id = s.subject_id
       AND l.study_id   = s.study_id

    LEFT JOIN visits v
        ON l.visit_id = v.visit_id

)

SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN matched_subject_id IS NULL
            THEN 1 ELSE 0
        END
    ) AS unknown_subject_or_study,

    SUM(
        CASE
            WHEN matched_visit_id IS NULL
            THEN 1 ELSE 0
        END
    ) AS unknown_visit_id,

    SUM(
        CASE
            WHEN matched_visit_id IS NOT NULL
             AND lab_subject_id <> visit_subject_id
            THEN 1 ELSE 0
        END
    ) AS visit_subject_mismatch,

    SUM(
        CASE
            WHEN matched_visit_id IS NOT NULL
             AND lab_study_id <> visit_study_id
            THEN 1 ELSE 0
        END
    ) AS visit_study_mismatch

FROM checked;

### Result

Referential-integrity analysis identified relationship inconsistencies in the Bronze laboratory-results feed.

Across **65,006 laboratory-result records**:

- **1,032** records do not resolve to a valid `(subject_id, study_id)` combination in the Bronze subject population.
- **0** records reference an unknown `visit_id`.
- **374** records reference a valid visit whose `subject_id` does not match the `subject_id` recorded on the laboratory result.
- **0** records show a study mismatch between the laboratory result and its referenced visit.

Therefore, every laboratory result references an existing visit, but subject-level attribution is not fully reliable.

The **374 visit-subject mismatches** are particularly important because the referenced visit exists but belongs to a different subject than the subject recorded on the laboratory result.

### Conclusion

The laboratory-result feed has complete visit-level referential coverage, but subject-level referential integrity is not fully consistent.

Silver should validate laboratory relationships against the clinical entity hierarchy rather than accepting the source identifiers independently.

The following conditions should be treated as blocking DQ failures:

- laboratory `subject_id` does not resolve to a valid subject/study combination;
- referenced `visit_id` does not exist;
- referenced visit belongs to a different subject;
- referenced visit belongs to a different study.

Records failing these checks should be routed to quarantine because their clinical ownership cannot be established reliably.

The referenced visit should not be used to silently overwrite an inconsistent laboratory `subject_id`. The original source values must remain available in quarantine for investigation and auditability.

## 9. Lab Result ID History Across Source Files

### Question

Does the same `lab_result_id` appear across multiple source files?

### Purpose

Determine the ingestion and history-management pattern required for Silver laboratory results.

If each `lab_result_id` occurs only once across the observed source files, the feed behaves as an append-only laboratory event feed.

If the same `lab_result_id` appears in multiple files with changed values, the source behaves as a change feed and requires CDC/history-management logic.

This analysis therefore determines whether Silver laboratory results should use a straightforward append-oriented transformation or CDC processing.

In [0]:
%sql

-- ============================================================
-- 9. LAB RESULT ID HISTORY ACROSS SOURCE FILES
-- ============================================================

WITH lab_history AS (

    SELECT
        lab_result_id,
        COUNT(*) AS row_count,
        COUNT(DISTINCT _source_file_name) AS source_file_count

    FROM clinical_trial_intelligence.bronze.lab_results

    GROUP BY
        lab_result_id

)

SELECT
    COUNT(*) AS distinct_lab_result_ids,

    SUM(
        CASE
            WHEN row_count > 1
            THEN 1 ELSE 0
        END
    ) AS repeated_lab_result_ids,

    SUM(
        CASE
            WHEN source_file_count > 1
            THEN 1 ELSE 0
        END
    ) AS lab_result_ids_across_multiple_files,

    MAX(row_count) AS max_rows_per_lab_result,

    MAX(source_file_count) AS max_source_files_per_lab_result

FROM lab_history;

### Result

The Bronze laboratory-results feed contains **65,006 distinct `lab_result_id` values across 65,006 records**.

The history analysis identified:

- **0** repeated `lab_result_id` values.
- **0** `lab_result_id` values appearing across multiple source files.
- Maximum records per `lab_result_id`: **1**.
- Maximum source files per `lab_result_id`: **1**.

Therefore, each observed laboratory result occurs exactly once across the current 10-file Bronze dataset.

No evidence of corrections, updates, or multiple versions of the same `lab_result_id` was identified.

### Conclusion

Based on the currently observed source files, the laboratory-results feed behaves as an **append-only clinical event feed** rather than a CDC change feed.

Therefore, Silver `lab_results` does **not require SCD Type 2 history or AUTO CDC processing**.

Each valid source laboratory result can be transformed and appended to Silver while preserving:

**one `lab_result_id` → one laboratory-result record**

This differs from the Subjects feed, where the same business key appears across multiple incremental files and clinically meaningful changes require SCD Type 2 history.

The append-only conclusion is based on the currently observed files. Silver should still enforce or monitor `lab_result_id` uniqueness so that unexpected future reissuance of an existing identifier is detected rather than silently accepted.


## 10. Collection-Date Validation

### Question

Can all source `collection_date` values be reliably converted to a valid date using the observed laboratory date format?

### Purpose

Validate the source date representation before defining the Silver data contract.

Bronze currently stores `collection_date` as a string, while Silver should expose it as a proper `DATE` value.

This analysis determines whether the observed `dd-MMM-yyyy` format can parse all populated collection dates and identifies any malformed or unexpected date representations that would require quarantine.

In [0]:
%sql

WITH parsed AS (

    SELECT
        lab_result_id,
        collection_date,
        _source_file_name,

        TRY_TO_DATE(
            TRIM(collection_date),
            'dd-MMM-yyyy'
        ) AS parsed_collection_date

    FROM clinical_trial_intelligence.bronze.lab_results
)

SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN collection_date IS NULL
              OR TRIM(collection_date) = ''
            THEN 1 ELSE 0
        END
    ) AS missing_collection_date,

    SUM(
        CASE
            WHEN collection_date IS NOT NULL
             AND TRIM(collection_date) <> ''
             AND parsed_collection_date IS NULL
            THEN 1 ELSE 0
        END
    ) AS invalid_collection_date,

    COUNT(DISTINCT
        CASE
            WHEN collection_date IS NOT NULL
             AND TRIM(collection_date) <> ''
             AND parsed_collection_date IS NULL
            THEN _source_file_name
        END
    ) AS affected_source_files,

    MIN(parsed_collection_date) AS earliest_collection_date,
    MAX(parsed_collection_date) AS latest_collection_date

FROM parsed;

### Result

All **65,006 laboratory-result records** contain a populated `collection_date`.

Using the observed `dd-MMM-yyyy` source format:

- **0** records have a missing collection date.
- **0** populated values fail date conversion.
- **0** source files contain invalid collection-date values.
- Earliest observed collection date: **2025-04-05**.
- Latest observed collection date: **2026-09-07**.

Therefore, the observed Bronze laboratory feed has complete and consistently parseable collection-date values.

### Conclusion

The source `collection_date` can be safely standardized from its Bronze string representation to the Silver `DATE` data type using the observed `dd-MMM-yyyy` format.

No collection-date records currently require quarantine because of missing or malformed date values.

Silver should nevertheless use safe date conversion so that future malformed source values produce an identifiable DQ failure rather than causing the pipeline to fail.

Therefore:

**valid `dd-MMM-yyyy` collection date → convert to Silver DATE**

**missing or unparseable collection date → blocking DQ failure → quarantine**

## 11. Result and Reference-Range Validity

### Question

Are laboratory result values and reference ranges structurally valid for clinical interpretation?

### Purpose

Evaluate whether laboratory measurements contain the numeric information required for reliable Silver standardization and abnormality derivation.

This analysis checks for:

- missing result values;
- missing result units;
- missing reference-range boundaries;
- invalid reference ranges where the lower bound exceeds the upper bound; and
- result values that cannot be interpreted because essential measurement information is incomplete.

The findings will determine which measurement-level conditions should be treated as blocking DQ failures in Silver.

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE WHEN result_value IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_result_value,

    SUM(
        CASE
            WHEN result_unit IS NULL
              OR TRIM(result_unit) = ''
        THEN 1 ELSE 0 END
    ) AS missing_result_unit,

    SUM(
        CASE WHEN reference_low IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_reference_low,

    SUM(
        CASE WHEN reference_high IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_reference_high,

    SUM(
        CASE
            WHEN reference_low IS NOT NULL
             AND reference_high IS NOT NULL
             AND reference_low > reference_high
        THEN 1 ELSE 0 END
    ) AS invalid_reference_range,

    SUM(
        CASE
            WHEN result_value IS NULL
              OR result_unit IS NULL
              OR TRIM(result_unit) = ''
        THEN 1 ELSE 0 END
    ) AS incomplete_measurement_rows,

    COUNT(DISTINCT
        CASE
            WHEN result_value IS NULL
              OR result_unit IS NULL
              OR TRIM(result_unit) = ''
              OR reference_low IS NULL
              OR reference_high IS NULL
              OR reference_low > reference_high
            THEN _source_file_name
        END
    ) AS affected_source_files

FROM clinical_trial_intelligence.bronze.lab_results;

### Result

The Bronze laboratory-results feed contains **65,006 records**.

Measurement and reference-range validation identified:

- **692** records with a missing `result_value`.
- **519** records with a missing `result_unit`.
- **0** records with a missing `reference_low`.
- **0** records with a missing `reference_high`.
- **0** records where `reference_low > reference_high`.
- **1,211** records with an incomplete laboratory measurement.
- These incomplete measurements occur across **8 source files**.

The missing-value and missing-unit populations are mutually exclusive in the observed dataset, since the combined incomplete-measurement count equals 692 + 519 = 1,211.

Therefore, the reference ranges are structurally complete and valid across the entire dataset, while a limited subset of laboratory measurements lacks information required for reliable interpretation.

### Conclusion

The reported reference ranges are structurally reliable for Silver processing because every record contains both range boundaries and no invalid lower/upper range relationships were identified.

However, a laboratory measurement cannot be fully standardized and independently interpreted when either `result_value` or `result_unit` is missing.

Therefore, the **1,211 incomplete measurement records should be treated as blocking DQ failures and routed to quarantine**.

The Silver policy should be:

**complete result value + result unit + valid reference range → eligible for standardization**

**missing result value or result unit → quarantine**

Reference-range completeness does not currently require additional remediation rules, although the validations should remain in the pipeline to detect future source-quality changes.

## 12. Unit Conversion and Standardized Result Validation

### Question

Can laboratory results with recognized test/unit combinations be safely converted to their standard laboratory units using the reference-defined conversion factors?

### Purpose

Validate the reference-driven standardization logic before implementing it in the Silver pipeline.

The analysis verifies that recognized laboratory test/unit combinations resolve to a single standard unit and conversion factor after deterministic reference deduplication.

It also evaluates the resulting standardized measurement using:

`standardized_result_value = result_value × conversion_factor`

This ensures that Silver can standardize heterogeneous source units without hardcoded laboratory-specific conversion logic.

In [0]:
%sql

WITH unit_ref AS (

    SELECT
        UPPER(TRIM(lab_test_code)) AS lab_test_code,
        UPPER(TRIM(raw_unit)) AS normalized_raw_unit,

        FIRST(standard_unit) AS standard_unit,
        FIRST(conversion_factor) AS conversion_factor

    FROM clinical_trial_intelligence.silver.ref_unit

    GROUP BY
        UPPER(TRIM(lab_test_code)),
        UPPER(TRIM(raw_unit))
),

standardized AS (

    SELECT
        l.lab_result_id,
        UPPER(TRIM(l.lab_test_code)) AS lab_test_code,
        l.result_value,
        l.result_unit,

        u.standard_unit,
        u.conversion_factor,

        l.result_value * u.conversion_factor
            AS standardized_result_value

    FROM clinical_trial_intelligence.bronze.lab_results l

    LEFT JOIN unit_ref u

        ON UPPER(TRIM(l.lab_test_code))
            = u.lab_test_code

       AND UPPER(TRIM(l.result_unit))
            = u.normalized_raw_unit
)

SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT lab_result_id)
        AS distinct_lab_results,

    SUM(
        CASE
            WHEN result_value IS NOT NULL
             AND result_unit IS NOT NULL
             AND conversion_factor IS NULL
            THEN 1 ELSE 0
        END
    ) AS unmapped_complete_measurements,

    SUM(
        CASE
            WHEN conversion_factor IS NOT NULL
             AND conversion_factor <> 1
            THEN 1 ELSE 0
        END
    ) AS converted_measurements,

    SUM(
        CASE
            WHEN result_value IS NOT NULL
             AND conversion_factor IS NOT NULL
             AND standardized_result_value IS NULL
            THEN 1 ELSE 0
        END
    ) AS failed_standardizations

FROM standardized;

### Result

The standardized unit-conversion test preserved the expected laboratory-result grain:

- **65,006** total rows.
- **65,006** distinct `lab_result_id` values.
- **377** complete measurements could not be mapped to a recognized test/unit reference combination.
- **862** measurements require an actual numerical unit conversion where `conversion_factor <> 1`.
- **0** mapped measurements failed standardization.

The **377 unmapped complete measurements** correspond to the previously identified unknown `lab_test_code = 'XXX'` population.

For all recognized and mapped laboratory measurements, the reference-driven conversion logic successfully produced a standardized result without changing the one-row-per-`lab_result_id` grain.

### Conclusion

The reference-driven unit-standardization approach is suitable for the Silver laboratory-results pipeline.

Silver can derive:

`standardized_result_value = result_value × conversion_factor`

using the deterministic `(lab_test_code, normalized_raw_unit)` reference lookup.

The **862 measurements requiring non-unit conversion factors** can therefore be standardized without laboratory-specific hardcoded transformation logic.

No standardization failures were observed for measurements that successfully resolved to the unit reference.

The **377 unmapped complete measurements** should remain blocking DQ failures because their unknown laboratory test code prevents reliable clinical interpretation.

The deterministic reference lookup also preserves the required Silver grain:

**one source `lab_result_id` → one standardized laboratory-result record**

## 13. Final Data-Quality and Quarantine Baseline

### Question

What is the expected Silver-valid and quarantine population after applying all blocking data-quality rules identified during laboratory-result exploration?

### Purpose

Establish the final pre-implementation DQ baseline for the laboratory-results Silver pipeline.

This analysis combines the blocking conditions identified during exploration so that the expected valid and quarantined populations are known before `lab_results.py` is implemented.

The resulting baseline will later be compared with the actual Silver and quarantine outputs during pipeline validation.

In [0]:
%sql

WITH unit_ref AS (

    SELECT
        UPPER(TRIM(lab_test_code)) AS lab_test_code,
        UPPER(TRIM(raw_unit)) AS normalized_raw_unit,
        FIRST(conversion_factor) AS conversion_factor

    FROM clinical_trial_intelligence.silver.ref_unit

    GROUP BY
        UPPER(TRIM(lab_test_code)),
        UPPER(TRIM(raw_unit))
),

subjects AS (

    SELECT DISTINCT
        subject_id,
        study_id

    FROM clinical_trial_intelligence.bronze.edc_subjects

    WHERE subject_id IS NOT NULL
),

visits AS (

    SELECT DISTINCT
        visit_id,
        subject_id,
        study_id

    FROM clinical_trial_intelligence.bronze.edc_visits

    WHERE visit_id IS NOT NULL
),

checked AS (

    SELECT
        l.*,

        s.subject_id AS matched_subject_id,

        v.visit_id AS matched_visit_id,
        v.subject_id AS visit_subject_id,
        v.study_id AS visit_study_id,

        u.conversion_factor,

        TRY_TO_DATE(
            TRIM(l.collection_date),
            'dd-MMM-yyyy'
        ) AS parsed_collection_date

    FROM clinical_trial_intelligence.bronze.lab_results l

    LEFT JOIN subjects s
        ON l.subject_id = s.subject_id
       AND l.study_id = s.study_id

    LEFT JOIN visits v
        ON l.visit_id = v.visit_id

    LEFT JOIN unit_ref u
        ON UPPER(TRIM(l.lab_test_code)) = u.lab_test_code
       AND UPPER(TRIM(l.result_unit)) = u.normalized_raw_unit
),

dq AS (

    SELECT
        *,

        ARRAY_COMPACT(
            ARRAY(

                CASE
                    WHEN lab_result_id IS NULL
                    THEN 'missing_lab_result_id'
                END,

                CASE
                    WHEN subject_id IS NULL
                    THEN 'missing_subject_id'
                END,

                CASE
                    WHEN study_id IS NULL
                    THEN 'missing_study_id'
                END,

                CASE
                    WHEN visit_id IS NULL
                    THEN 'missing_visit_id'
                END,

                CASE
                    WHEN lab_test_code IS NULL
                    THEN 'missing_lab_test_code'
                END,

                CASE
                    WHEN matched_subject_id IS NULL
                    THEN 'unknown_subject_or_study'
                END,

                CASE
                    WHEN matched_visit_id IS NULL
                    THEN 'unknown_visit_id'
                END,

                CASE
                    WHEN matched_visit_id IS NOT NULL
                     AND subject_id <> visit_subject_id
                    THEN 'visit_subject_mismatch'
                END,

                CASE
                    WHEN matched_visit_id IS NOT NULL
                     AND study_id <> visit_study_id
                    THEN 'visit_study_mismatch'
                END,

                CASE
                    WHEN result_value IS NULL
                    THEN 'missing_result_value'
                END,

                CASE
                    WHEN result_unit IS NULL
                      OR TRIM(result_unit) = ''
                    THEN 'missing_result_unit'
                END,

                CASE
                    WHEN reference_low IS NULL
                    THEN 'missing_reference_low'
                END,

                CASE
                    WHEN reference_high IS NULL
                    THEN 'missing_reference_high'
                END,

                CASE
                    WHEN reference_low IS NOT NULL
                     AND reference_high IS NOT NULL
                     AND reference_low > reference_high
                    THEN 'invalid_reference_range'
                END,

                CASE
                    WHEN collection_date IS NULL
                      OR TRIM(collection_date) = ''
                      OR parsed_collection_date IS NULL
                    THEN 'invalid_collection_date'
                END,

                CASE
                    WHEN result_value IS NOT NULL
                     AND result_unit IS NOT NULL
                     AND TRIM(result_unit) <> ''
                     AND conversion_factor IS NULL
                    THEN 'unmapped_test_unit'
                END
            )
        ) AS dq_failures

    FROM checked
)

SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE WHEN SIZE(dq_failures) = 0
        THEN 1 ELSE 0 END
    ) AS expected_silver_rows,

    SUM(
        CASE WHEN SIZE(dq_failures) > 0
        THEN 1 ELSE 0 END
    ) AS expected_quarantine_rows,

    SUM(
        CASE WHEN SIZE(dq_failures) > 1
        THEN 1 ELSE 0 END
    ) AS rows_with_multiple_failures,

    MAX(SIZE(dq_failures))
        AS max_failures_per_row

FROM dq;

### Result

The combined data-quality assessment was applied across all **65,006 Bronze laboratory-result records** using the blocking rules established during exploration.

The expected processing outcome is:

- **62,401 records** eligible for the Silver `lab_results` table.
- **2,605 records** expected to be routed to quarantine.
- **389 records** violate more than one blocking DQ rule.
- The maximum number of blocking failures observed on a single record is **2**.

Therefore, approximately **96.0%** of the Bronze laboratory-result population is expected to pass the identified Silver quality requirements, while approximately **4.0%** requires quarantine.

The presence of 389 records with multiple failures also confirms that quarantine must retain all applicable DQ failure reasons rather than assigning only a single failure category to each record.

### Conclusion

The laboratory-results exploration has established the final transformation and data-quality design required for Silver.

Based on the currently observed source files, the laboratory-results feed behaves as an **append-only clinical event feed**. No repeated `lab_result_id` values or cross-file versions were identified; therefore, SCD Type 2 and AUTO CDC processing are not required.

Silver processing should:

- preserve one record per `lab_result_id`;
- standardize laboratory identifiers and descriptive fields;
- convert `collection_date` to a proper `DATE`;
- resolve laboratory tests through the laboratory-test reference;
- resolve and standardize units through a deterministic unit-reference lookup;
- derive `standardized_result_value` using the reference-defined conversion factor;
- derive abnormality from the standardized measurement and applicable reference range;
- retain the source-provided abnormal flag separately for lineage and discrepancy analysis;
- validate subject, visit, and study relationships;
- route records with blocking DQ failures to quarantine; and
- retain all DQ failure reasons when a record violates multiple rules.

The expected implementation baseline is:

**Bronze: 65,006**

→ **Silver-valid: 62,401**

→ **Quarantine: 2,605**

These values become the validation baseline for the implemented `lab_results.py` pipeline and should be verified after pipeline execution.

The exploration phase for `lab_results` is therefore complete.

## 14. Unit-Mapping Duplicate Conflict Validation

### Question

Do duplicate normalized unit-mapping keys contain conflicting standard units or conversion factors?

### Purpose

Validate whether duplicate reference rows are semantically identical before collapsing them into a single deterministic mapping.

The Silver pipeline must not use an arbitrary `FIRST()` value when multiple reference rows exist for the same normalized `(lab_test_code, raw_unit)` key.

If all duplicate rows agree on both `standard_unit` and `conversion_factor`, the reference can be safely collapsed deterministically.

If conflicting values exist, the reference data itself is invalid and must be corrected before Silver standardization.

In [0]:
%sql

WITH normalized_ref AS (

    SELECT
        UPPER(TRIM(lab_test_code)) AS lab_test_code,

        UPPER(
            REGEXP_REPLACE(
                TRIM(raw_unit),
                '\\s+',
                ''
            )
        ) AS normalized_raw_unit,

        standard_unit,
        conversion_factor

    FROM clinical_trial_intelligence.bronze.ref_unit_mapping
),

conflict_check AS (

    SELECT
        lab_test_code,
        normalized_raw_unit,

        COUNT(*) AS reference_rows,

        COUNT(DISTINCT standard_unit)
            AS distinct_standard_units,

        COUNT(DISTINCT conversion_factor)
            AS distinct_conversion_factors

    FROM normalized_ref

    GROUP BY
        lab_test_code,
        normalized_raw_unit
)

SELECT *
FROM conflict_check
WHERE reference_rows > 1
ORDER BY
    lab_test_code,
    normalized_raw_unit;

### Result

Duplicate normalized unit-mapping keys were identified for six laboratory tests:

- ALT / U/L: 2 reference rows
- AST / U/L: 2 reference rows
- CREAT / MG/DL: 4 reference rows
- GLUC / MG/DL: 3 reference rows
- HGB / G/DL: 3 reference rows
- K / MMOL/L: 2 reference rows

For every duplicate normalized key:

- `distinct_standard_units = 1`
- `distinct_conversion_factors = 1`

Therefore, the duplicate reference rows are semantically consistent. No conflicting standard-unit or conversion-factor mappings were identified.

### Conclusion

Normalization of raw laboratory units creates duplicate reference keys because multiple source spellings collapse to the same normalized representation.

However, all duplicate keys resolve to exactly one standard unit and one conversion factor. The duplication is therefore a reference-cardinality issue rather than a reference-value conflict.

Silver may safely collapse these duplicate mappings before joining them to laboratory results.

The collapse must still be deterministic. `FIRST()` should not be used for this purpose because its result is not guaranteed to be deterministic after a shuffle. Since the duplicate values have been demonstrated to be semantically identical, deterministic aggregation using `MAX(standard_unit)` and `MAX(conversion_factor)` is appropriate.

This prevents reference-table fan-out while preserving the validated unit-standardization semantics.